In [65]:
from pathlib import Path
import getpass
import pandas as pd
from sqlalchemy import create_engine, text

df = pd.read_csv(Path("../data/raw/Steel_industry_data.csv"))
df.columns

Index(['date', 'Usage_kWh', 'Lagging_Current_Reactive.Power_kVarh',
       'Leading_Current_Reactive_Power_kVarh', 'CO2(tCO2)',
       'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor', 'NSM',
       'WeekStatus', 'Day_of_week', 'Load_Type'],
      dtype='str')

In [66]:
df["Load_Type"].unique()

<StringArray>
['Light_Load', 'Medium_Load', 'Maximum_Load']
Length: 3, dtype: str

In [67]:
rename = {
    "date": "ts",
    "Usage_kWh": "usage_kwh",
    "Lagging_Current_Reactive.Power_kVarh": "lagging_reactive_kvarh",
    "Leading_Current_Reactive_Power_kVarh": "leading_reactive_kvarh",
    "CO2(tCO2)": "co2_t",
    "Lagging_Current_Power_Factor": "lagging_pf",
    "Leading_Current_Power_Factor": "leading_pf",
    "NSM": "nsm",
    "WeekStatus": "week_status",
    "Day_of_week": "day_of_week",
    "Load_Type": "load_type",
}
df = df.rename(columns=rename)
df["ts"] = pd.to_datetime(df["ts"], format='%d/%m/%Y %H:%M')
df.dtypes

ts                        datetime64[us]
usage_kwh                        float64
lagging_reactive_kvarh           float64
leading_reactive_kvarh           float64
co2_t                            float64
lagging_pf                       float64
leading_pf                       float64
nsm                                int64
week_status                          str
day_of_week                          str
load_type                            str
dtype: object

In [68]:
user = getpass.getuser()
engine = create_engine(f"postgresql+psycopg://{user}@localhost:5432/energy_kpi")

df.to_sql("raw_steel", engine, if_exists="replace", index=False)

with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM raw_steel")).scalar())

35040


In [69]:
ratio = df["co2_t"] / df["usage_kwh"]
ratio[df["usage_kwh"] > 0].describe()

count    35039.000000
mean         0.000185
std          0.000231
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000440
max          0.000917
dtype: float64

In [70]:
df.set_index('ts', inplace = True)
pd.crosstab(df.index.hour, df["load_type"])


load_type,Light_Load,Maximum_Load,Medium_Load
row_0,,,
0,1460,0,0
1,1460,0,0
2,1460,0,0
3,1460,0,0
4,1460,0,0
5,1460,0,0
6,1460,0,0
7,1460,0,0
8,1460,0,0
